# Fitting Vs Validation

Display the fitting and validation metrics in `loss` and `accuracy` graphs per client.

Assuming that this notebook will be used after having completed a training run using the framework, it will fetch the fitting and validation metrics from the server-side Minio datastore. For this to work, place a file named `scenario.json` in the same directory as this notebook containing the scenario that has been used to train the model with.

Set the values inherent to your setup in the `Constants` section.

### Imports

In [ ]:
import json
import matplotlib.pyplot as plt
import os
from minio import Minio

### Constants

If the INFERENCE_DATASET(.parquet) is not found in this folder, the notebook will try to download it from Kaggle according to this format: [username]-[dataset]

In [ ]:
MINIO_SERVER_ENDPOINT = "[server-ip]:9000" # change with your server's IP
MINIO_SERVER_ACCESS_KEY = "aide-fl1"
MINIO_SERVER_SECRET_KEY = "aide-fl1"
MINIO_SERVER_METRICS_BUCKET = "metrics"

### Load scenario

In [ ]:
with open("scenario.json", "r") as f:
    scenario = json.load(f)

scenario_name = scenario["name"]

### Download metrics

In [ ]:
metrics_path = f"{scenario_name}_fitted_validated_metrics.json"

minio_client = Minio(MINIO_SERVER_ENDPOINT, access_key=MINIO_SERVER_ACCESS_KEY, secret_key=MINIO_SERVER_SECRET_KEY, secure=False)
minio_client.fget_object(MINIO_SERVER_METRICS_BUCKET, metrics_path, metrics_path)

### Fitting Vs Validation

In [ ]:
def plot_knit_metrics(history, save=False, outdir="plots"):
    """
    Stitches metrics across rounds into continuous plots per client.
    Shows 2 graphs per client: Accuracy and Loss.
    Adds vertical dashed lines to mark round boundaries.
    """

    if save:
        os.makedirs(outdir, exist_ok=True)

    # Collect data per client across rounds
    client_histories = {}

    for round_num, clients in history.items():
         for i in range(len(clients)):
            client_data = clients[i]
            client_name = i
            
            if client_name not in client_histories:
                client_histories[client_name] = {
                    "accuracy": [],
                    "val_accuracy": [],
                    "loss": [],
                    "val_loss": [],
                    "round_breaks": []
                }

            # Append metrics
            if client_histories[client_name].get("accuracy"):
                client_histories[client_name]["accuracy"].extend(client_data["accuracy"])
                client_histories[client_name]["val_accuracy"].extend(client_data["val_accuracy"])
            client_histories[client_name]["loss"].extend(client_data["loss"])
            client_histories[client_name]["val_loss"].extend(client_data["val_loss"])
            
            # Track where this round ends
            total_epochs_so_far = len(client_histories[client_name]["loss"])
            client_histories[client_name]["round_breaks"].append(total_epochs_so_far)

    # Plot per client
    for client_name, metrics in client_histories.items():
        epochs = range(1, len(metrics["loss"]) + 1)

        if client_histories[client_name].get("accuracy"):
            # Accuracy plot
            plt.figure(figsize=(8, 4))
            plt.plot(epochs, metrics["accuracy"], label="Training Accuracy")
            plt.plot(epochs, metrics["val_accuracy"], label="Validation Accuracy")
            for rb in metrics["round_breaks"][:-1]:  # skip final break
                plt.axvline(rb, color="green", linestyle="--", alpha=0.6)
            plt.title(f"Client {client_name} - Accuracy")
            plt.xlabel("Epochs (stitched across rounds)")
            plt.ylabel("Accuracy")
            plt.legend()
            plt.grid(True)
            plt.tight_layout()
            if save:
                plt.savefig(f"{outdir}/client_{client_name}_accuracy.png")
                plt.close()
            else:
                plt.show()

        # Loss plot
        plt.figure(figsize=(8, 4))
        plt.plot(epochs, metrics["loss"], label="Training Loss")
        plt.plot(epochs, metrics["val_loss"], label="Validation Loss")
        for rb in metrics["round_breaks"][:-1]:
            plt.axvline(rb, color="green", linestyle="--", alpha=0.6)
        plt.title(f"Client {client_name} - Loss")
        plt.xlabel("Epochs (stitched across rounds)")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        if save:
            plt.savefig(f"{outdir}/client_{client_name}_loss.png")
            plt.close()
        else:
            plt.show()


with open(metrics_path) as f:
    history = json.load(f)
    
plot_knit_metrics(history)